<a href="https://colab.research.google.com/github/andrewemmanuelyohanson31-ship-it/MLClothing/blob/main/Welcome_to_Colab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/alexeygrigorev/clothing-dataset-small.git

fatal: destination path 'clothing-dataset-small' already exists and is not an empty directory.


In [ ]:
!pip install scikit-image
!pip install streamlit
!pip install pyngrok

In [ ]:
import os
import cv2
import numpy as np

from skimage.feature import hog

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

import joblib

In [ ]:
train_dir = "clothing-dataset-small/train"
val_dir = "clothing-dataset-small/validation"

classes = os.listdir(train_dir)

print(classes)

['hat', 'shoes', 'shorts', 'longsleeve', 't-shirt', 'shirt', 'dress', 'outwear', 'pants', 'skirt']


In [ ]:
def extract_features(image_path):

    img = cv2.imread(image_path)

    img = cv2.resize(img, (128, 128))

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8,8),
        cells_per_block=(2,2),
        transform_sqrt=True,
        block_norm='L2-Hys'
    )

    return features

In [ ]:
X_train = []
y_train = []

for label in classes:

    folder = os.path.join(train_dir, label)

    for file in os.listdir(folder):

        path = os.path.join(folder, file)

        try:
            features = extract_features(path)

            X_train.append(features)
            y_train.append(label)

        except:
            pass

print("Training data loaded")

Training data loaded


In [ ]:
X_val = []
y_val = []

for label in classes:

    folder = os.path.join(val_dir, label)

    for file in os.listdir(folder):

        path = os.path.join(folder, file)

        try:
            features = extract_features(path)

            X_val.append(features)
            y_val.append(label)

        except:
            pass

print("Validation data loaded")

Validation data loaded


In [ ]:
model = SVC(kernel='linear')

model.fit(X_train, y_train)

print("Model trained")

Model trained


In [ ]:
predictions = model.predict(X_val)

accuracy = accuracy_score(y_val, predictions)

print("Accuracy:", accuracy)

Accuracy: 0.6715542521994134


In [ ]:
joblib.dump(model, "svm_model.pkl")

print("Model saved")

Model saved


In [ ]:
!mkdir -p models
!mkdir -p crops/top
!mkdir -p crops/bottom

In [ ]:
%%writefile classifier.py

import numpy as np
import cv2
import joblib

from skimage.feature import hog

# -----------------------------------
# TOP & BOTTOM CLASSES
# -----------------------------------

TOP_CLASSES = [
    "dress",
    "longsleeve",
    "outwear",
    "shirt",
    "t-shirt"
]

BOTTOM_CLASSES = [
    "pants",
    "shorts",
    "skirt"
]

# -----------------------------------
# LOAD MODELS
# -----------------------------------

def load_models():

    top_model = joblib.load(
        "models/top_model.pkl"
    )

    bottom_model = joblib.load(
        "models/bottom_model.pkl"
    )

    return top_model, bottom_model

# -----------------------------------
# PREPROCESS IMAGE
# -----------------------------------

def preprocess_image(image):

    img = np.array(image)

    img = cv2.resize(
        img,
        (128, 128)
    )

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_RGB2GRAY
    )

    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8,8),
        cells_per_block=(2,2),
        transform_sqrt=True,
        block_norm='L2-Hys'
    )

    return features

# -----------------------------------
# CLASSIFY TOP
# -----------------------------------

def classify_top(model, image):

    features = preprocess_image(image)

    prediction = model.predict(
        [features]
    )[0]

    probabilities = model.predict_proba(
        [features]
    )[0]

    confidence = np.max(probabilities) * 100

    return prediction, confidence

# -----------------------------------
# CLASSIFY BOTTOM
# -----------------------------------

def classify_bottom(model, image):

    features = preprocess_image(image)

    prediction = model.predict(
        [features]
    )[0]

    probabilities = model.predict_proba(
        [features]
    )[0]

    confidence = np.max(probabilities) * 100

    return prediction, confidence

Overwriting classifier.py


In [ ]:
%%writefile utils.py

import os

# -----------------------------
# Create Folder
# -----------------------------

def ensure_folder(path):

    folder = os.path.dirname(path)

    os.makedirs(folder, exist_ok=True)

# -----------------------------
# Save Image
# -----------------------------

def save_image(image, path):

    ensure_folder(path)

    image.save(path)

# -----------------------------
# Resize Image
# -----------------------------

def resize_image(image, size=(224, 224)):

    return image.resize(size)

Overwriting utils.py


In [ ]:
!pip install streamlit
!pip install streamlit-cropper
!pip install pyngrok

In [ ]:
%%writefile app.py

import streamlit as st

from PIL import Image

from streamlit_cropper import st_cropper

from classifier import (
    load_models,
    classify_top,
    classify_bottom
)

from utils import (
    save_image,
    resize_image
)

# -----------------------------
# Page Config
# -----------------------------

st.set_page_config(
    page_title="Clothing Classifier",
    layout="wide"
)

st.title("Clothing Classifier")

# -----------------------------
# Load Models
# -----------------------------

top_model, bottom_model = load_models()

# -----------------------------
# Upload Image
# -----------------------------

uploaded_file = st.file_uploader(
    "Upload Image",
    type=["jpg", "jpeg", "png"]
)

camera_image = st.camera_input(
    "Or Capture Image"
)

image = None

if uploaded_file:

    image = Image.open(uploaded_file).convert("RGB")

elif camera_image:

    image = Image.open(camera_image).convert("RGB")

# -----------------------------
# Main App
# -----------------------------

if image is not None:

    st.image(image)

    col1, col2 = st.columns(2)

    # -----------------------------
    # TOP Crop
    # -----------------------------

    with col1:

        st.subheader("TOP Clothing")

        cropped_top = st_cropper(
            image,
            realtime_update=True,
            key="top_cropper"
        )

        st.image(cropped_top)

        save_image(
            cropped_top,
            "crops/top/top_crop.png"
        )

    # -----------------------------
    # BOTTOM Crop
    # -----------------------------

    with col2:

        st.subheader("BOTTOM Clothing")

        cropped_bottom = st_cropper(
            image,
            realtime_update=True,
            key="bottom_cropper"
        )

        st.image(cropped_bottom)

        save_image(
            cropped_bottom,
            "crops/bottom/bottom_crop.png"
        )

    # -----------------------------
    # Analyze Button
    # -----------------------------

    if st.button("Analyze Clothing"):

        top_resized = resize_image(cropped_top)

        bottom_resized = resize_image(cropped_bottom)

        top_label, top_conf = classify_top(
            top_model,
            top_resized
        )

        bottom_label, bottom_conf = classify_bottom(
            bottom_model,
            bottom_resized
        )

        st.divider()

        result1, result2 = st.columns(2)

        with result1:

            st.success("Top Clothing")

            st.write(
                f"{top_label} ({top_conf:.2f}%)"
            )

        with result2:

            st.success("Bottom Clothing")

            st.write(
                f"{bottom_label} ({bottom_conf:.2f}%)"
            )

Overwriting app.py


In [ ]:
import os
import cv2
import numpy as np

from skimage.feature import hog

from sklearn.svm import SVC

import joblib

In [ ]:
train_dir = "clothing-dataset-small/train"

all_classes = os.listdir(train_dir)

print(all_classes)

['hat', 'shoes', 'shorts', 'longsleeve', 't-shirt', 'shirt', 'dress', 'outwear', 'pants', 'skirt']


In [ ]:
TOP_CLASSES = [
    "dress",
    "longsleeve",
    "outwear",
    "shirt",
    "t-shirt"
]

BOTTOM_CLASSES = [
    "pants",
    "shorts",
    "skirt"
]

In [ ]:
def extract_features(image_path):

    img = cv2.imread(image_path)

    img = cv2.resize(img, (128, 128))

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8,8),
        cells_per_block=(2,2),
        transform_sqrt=True,
        block_norm='L2-Hys'
    )

    return features

In [ ]:
X_top = []
y_top = []

for label in TOP_CLASSES:

    folder = os.path.join(
        train_dir,
        label
    )

    for file in os.listdir(folder):

        path = os.path.join(
            folder,
            file
        )

        try:

            features = extract_features(path)

            X_top.append(features)

            y_top.append(label)

        except:
            pass

print("TOP dataset loaded")

TOP dataset loaded


In [ ]:
top_model = SVC(
    kernel='linear',
    probability=True
)

top_model.fit(
    X_top,
    y_top
)

print("TOP model trained")

TOP model trained


In [ ]:
joblib.dump(
    top_model,
    "models/top_model.pkl"
)

print("TOP model saved")

TOP model saved


In [ ]:
X_bottom = []
y_bottom = []

for label in BOTTOM_CLASSES:

    folder = os.path.join(
        train_dir,
        label
    )

    for file in os.listdir(folder):

        path = os.path.join(
            folder,
            file
        )

        try:

            features = extract_features(path)

            X_bottom.append(features)

            y_bottom.append(label)

        except:
            pass

print("BOTTOM dataset loaded")

BOTTOM dataset loaded


In [ ]:
bottom_model = SVC(
    kernel='linear',
    probability=True
)

bottom_model.fit(
    X_bottom,
    y_bottom
)

print("BOTTOM model trained")

BOTTOM model trained


In [ ]:
joblib.dump(
    bottom_model,
    "models/bottom_model.pkl"
)

print("BOTTOM model saved")

BOTTOM model saved


In [ ]:
!ngrok config add-authtoken 3DtI7XQq0FojVI26yRtK1RP9dza_4hhH1iyGUTjjEkeiacADn

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from pyngrok import ngrok

ngrok.kill()

In [ ]:
from pyngrok import ngrok
import threading
import os

def run_streamlit():
    os.system("streamlit run app.py --server.port 8501")

threading.Thread(target=run_streamlit).start()

public_url = ngrok.connect(8501)

print(public_url)

PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://macaw-craftsman-badass.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

TOP_CLASSES = [
    "dress",
    "longsleeve",
    "outwear",
    "shirt",
    "t-shirt"
]

BOTTOM_CLASSES = [
    "pants",
    "shorts",
    "skirt"
]

X_val_top = []
y_val_top = []
X_val_bottom = []
y_val_bottom = []

for i, label in enumerate(y_val):
    if label in TOP_CLASSES:
        X_val_top.append(X_val[i])
        y_val_top.append(label)
    elif label in BOTTOM_CLASSES:
        X_val_bottom.append(X_val[i])
        y_val_bottom.append(label)

print("\nTOP")
if len(X_val_top) > 0:
    predictions_top = top_model.predict(X_val_top)
    print("Classification Report (TOP Model):")
    print(classification_report(y_val_top, predictions_top, target_names=TOP_CLASSES, zero_division=0))
    print("Confusion Matrix (TOP Model):")
    print(confusion_matrix(y_val_top, predictions_top, labels=TOP_CLASSES))
else:
    print("No validation data available for TOP classes.")

print("\nBOTTOM")
if len(X_val_bottom) > 0:
    predictions_bottom = bottom_model.predict(X_val_bottom)
    print("Classification Report (BOTTOM Model):")
    print(classification_report(y_val_bottom, predictions_bottom, target_names=BOTTOM_CLASSES, zero_division=0))
    print("Confusion Matrix (BOTTOM Model):")
    print(confusion_matrix(y_val_bottom, predictions_bottom, labels=BOTTOM_CLASSES))
else:
    print("No validation data available for BOTTOM classes.")


TOP
Classification Report (TOP Model):
              precision    recall  f1-score   support

       dress       0.72      0.56      0.63        32
  longsleeve       0.58      0.57      0.58        49
     outwear       0.62      0.54      0.58        24
       shirt       0.56      0.52      0.54        29
     t-shirt       0.80      0.93      0.86        81

    accuracy                           0.69       215
   macro avg       0.66      0.62      0.64       215
weighted avg       0.68      0.69      0.69       215

Confusion Matrix (TOP Model):
[[18  5  2  2  5]
 [ 4 28  3  6  8]
 [ 0  5 13  4  2]
 [ 2  5  3 15  4]
 [ 1  5  0  0 75]]

BOTTOM
Classification Report (BOTTOM Model):
              precision    recall  f1-score   support

       pants       0.89      0.96      0.92        49
      shorts       0.76      0.76      0.76        25
       skirt       0.88      0.58      0.70        12

    accuracy                           0.85        86
   macro avg       0.84      0.7

In [ ]:
def classify_top(model, image):

    features = preprocess_image(image)

    probs = model.predict_proba([features])[0]
    classes = model.classes_

    # ambil top 3
    top3_idx = np.argsort(probs)[::-1][:3]

    top3 = [
        (classes[i], probs[i] * 100)
        for i in top3_idx
    ]

    prediction = top3[0][0]
    confidence = top3[0][1]

    return prediction, confidence, top3

In [ ]:
def classify_bottom(model, image):

    features = preprocess_image(image)

    probs = model.predict_proba([features])[0]
    classes = model.classes_

    top3_idx = np.argsort(probs)[::-1][:3]

    top3 = [
        (classes[i], probs[i] * 100)
        for i in top3_idx
    ]

    prediction = top3[0][0]
    confidence = top3[0][1]

    return prediction, confidence, top3